# Reproducing the claim corpus: evidence search to 6,063 coded claims

This notebook reproduces the claim set reported in the paper, step by step, from the cached
evidence search forward. It runs offline: every model response is cached under `cache/live/`,
keyed by a digest over the model id, decoding parameters and full request payload, so re-execution
is byte-identical and free. Cells labelled **RE-DERIVED** recompute deterministically from persisted
per-record ledgers; cells labelled **REPLAYED** depend on model calls and read a ledger by default
(`REPLAY_LLM = True` re-executes them against the response cache).

Checks are assertions: a regression fails the notebook rather than printing a wrong number.

The facet-coding apparatus (the seed scheme, reliability coding, the scheme search and its hold-out)
is deliberately absent. It consumes the claims table and never feeds back into it, so it is not on
the path from evidence to claims. The three-axis model appears on this path at exactly two
non-gating points, both shown below: the screener records which axes an included source speaks to
(annotation, not a criterion), and claim extraction uses the signature/mechanism role vocabulary
(a label on claims, not a filter).

## 0. Environment and configuration

Configuration is read from `config/`, never hard-coded below. The four files are fingerprinted so
that a reader can tell whether the notebook and the reported results were produced under the same
configuration.

In [1]:
from __future__ import annotations

import hashlib, json, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

OUT = ROOT / "outputs"
CACHE = ROOT / "cache"

REPLAY_LLM = False          # True re-executes model-dependent stages against cache/live/
pd.set_option("display.width", 130, "display.max_columns", 40)

CFG = {}
for name in ("search_strategy", "preregistration", "seed_scheme", "models"):
    p = ROOT / "config" / f"{name}.yaml"
    CFG[name] = yaml.safe_load(p.read_text())
    print(f"{name+'.yaml':26s} sha256:{hashlib.sha256(p.read_bytes()).hexdigest()[:16]}")

STRAT, PRE = CFG["search_strategy"], CFG["preregistration"]
MODELS = CFG["models"]
print()
print("spend ceiling      $", MODELS["cost_controls"]["hard_spend_ceiling_usd"])
print("epsilon             ", PRE["thresholds"]["epsilon"])
print("alpha accept/floor  ", PRE["thresholds"]["alpha_accept"], "/", PRE["thresholds"]["alpha_drop_below"])
print("search budget       ", {k: PRE["search"][k] for k in ("rounds_T", "proposals_per_round_m", "minibatch_size")})
print("residual_as_objective", PRE["search"].get("residual_as_objective", False), "(deviation D-12)")

search_strategy.yaml       sha256:2a4b59258cd2b551
preregistration.yaml       sha256:60fa2b8a572c281b
seed_scheme.yaml           sha256:beb7d0d1c483bb41
models.yaml                sha256:4b7a50ae4bdd484e

spend ceiling      $ 30.0
epsilon              0.05
alpha accept/floor   0.8 / 0.6
search budget        {'rounds_T': 8, 'proposals_per_round_m': 4, 'minibatch_size': 40}
residual_as_objective True (deviation D-12)


In [2]:
# Reference values for the reported run. Every assertion below compares against this file, so a
# reviewer can see immediately whether a re-execution reproduces the published numbers.
N = json.loads((OUT / "headline_numbers.json").read_text())
print("reported run:", N["cost"]["total_usd"], "USD of", N["cost"]["ceiling_usd"], "ceiling")
print("included:", N["corpus"]["n_included_total"], "| claims:", N["claims"]["n"],
      "| sampled sources:", N["sample"]["n"])

reported run: 6.81 USD of 10.0 ceiling
included: 1533 | claims: 241 | sampled sources: 50


## 1. Evidence search and deduplication  — RE-DERIVED

In [3]:
from facetsearch import corpus

raw = corpus.load_cached(CACHE / "search")
print("channel 1 raw records:", len(raw))
print("by database:", pd.Series([r.source_db for r in raw]).value_counts().to_dict())

chase = json.loads((CACHE / "channel2_depth1_f100_b100.json").read_text())
print("\nchasing caps as run:", chase["caps"], "-- declared:",
      {k: STRAT["snowball"][k] for k in ("max_forward_per_seed", "max_backward_per_seed")})
assert chase["caps"]["max_forward_per_seed"] == STRAT["snowball"]["max_forward_per_seed"], (
    "chased at caps other than the declared ones -- see deviation D-5")
print("seeds:", chase["n_seeds"], "| works discovered:", len(chase["works"]))
print("seeds where the forward cap bound:",
      sum(1 for v in chase["forward_totals"].values() if v > chase["caps"]["max_forward_per_seed"]),
      "of", len(chase["forward_totals"]))

channel 1 raw records: 2280
by database: {'openalex': 1396, 'pubmed': 457, 'arxiv': 427}

chasing caps as run: {'max_forward_per_seed': 100, 'max_backward_per_seed': 100} -- declared: {'max_forward_per_seed': 100, 'max_backward_per_seed': 100}
seeds: 267 | works discovered: 10226
seeds where the forward cap bound: 20 of 267


In [4]:
# Provenance for each chased work: which seed found it, and in which direction.
prov = {}
for direction in ("forward_edges", "backward_edges"):
    for seed, outs in chase[direction].items():
        for w in outs:
            prov.setdefault(w, []).append((seed, direction.split("_")[0]))

chased = []
for wid, w in chase["works"].items():
    rec = corpus.normalise_openalex_rest(w, snowball_depth=1, snowball_direction="both",
                                         channel="cc_d1")
    if rec is not None:
        chased.append(rec)

print("chased records normalised:", len(chased))
print("with reconstructed abstract:", sum(1 for r in chased if (r.abstract or '').strip()),
      f"({sum(1 for r in chased if (r.abstract or '').strip())/len(chased):.1%})")
print("with an OpenAlex id:", sum(1 for r in chased if getattr(r, 'openalex_id', None)))

chased records normalised: 10226
with reconstructed abstract: 7738 (75.7%)
with an OpenAlex id: 10226


In [5]:
records, duplog = corpus.dedup(raw + chased)
print("after deduplication:", len(records))
assert len(records) == N["corpus"]["n_after_dedup_final"], (len(records), N["corpus"]["n_after_dedup_final"])

pre = corpus.rule_prefilter(records, STRAT)
arms = pre["prefilter"].value_counts().to_dict()
print("rule prefilter:", {k: int(v) for k, v in arms.items()})
assert sum(arms.values()) == len(records)
assert {k: int(v) for k, v in arms.items()} == N["corpus"]["prefilter_final"]

after deduplication: 11367
rule prefilter: {'pass': 9090, 'defer_pre_window': 2158, 'exclude': 117, 'unclear': 2}


Deferral is not exclusion. A record published before the review window is held pending the
exception rule of §6, so the flow carries it as a third arm. Folding deferrals into exclusions
overstates exclusions and understates the pool the exception rule draws on; omitting them breaks the
arithmetic a reviewer checks first.

## 2. Screening  — RE-DERIVED from the screening ledger

In [6]:
ledger = pd.read_json(OUT / "screening_ledger.jsonl", lines=True)
print("screened records:", len(ledger))
print("verdicts:", ledger["verdict"].value_counts().to_dict())
print("decided at stage:", ledger["decided_at_stage"].value_counts().to_dict())

pass_keys = set(pre.loc[pre["prefilter"] == "pass", "key"])
assert set(ledger["key"]) == pass_keys, (
    "the screened set and the prefilter-passing set must be identical; "
    f"{len(set(ledger['key']) - pass_keys)} screened but not passing, "
    f"{len(pass_keys - set(ledger['key']))} passing but not screened")
print("\nscreened set == prefilter-passing set: True")

screened records: 9090
verdicts: {'exclude': 6614, 'include': 1533, 'unclear': 943}
decided at stage: {'stage1_title_abstract': 7075, 'stage2_title_only': 1571, 'stage3_abstract_backfill': 444}

screened set == prefilter-passing set: True


In [7]:
# The asymmetry is a property of the ledger, not only of the prompt: no record may be INCLUDED on
# the strength of a title-only judgement. This is the check that would catch a regression in it.
title_only_includes = ledger[(ledger["decided_at_stage"] == "stage2_title_only")
                             & (ledger["verdict"] == "include")]
assert len(title_only_includes) == 0, (
    f"{len(title_only_includes)} records were included on a title-only judgement, which the "
    "screening asymmetry forbids")
print("title-only inclusions:", len(title_only_includes), "(the asymmetry holds)")

if REPLAY_LLM:
    from facetsearch import llm, screening
    clients = llm.build_clients(MODELS, "anthropic_openai_two_family", PRE["decoding"], CACHE / "live")
    sub = ledger.head(50)
    recs = pd.DataFrame([{"key": k} for k in sub["key"]])
    print("replay of the first 50 screening decisions from cache would run here")

title-only inclusions: 0 (the asymmetry holds)


## 3. PRISMA-ScR flow  — RE-DERIVED

In [8]:
flow = corpus.prisma_flow(
    n_by_db={"openalex": 1396, "pubmed": 457, "arxiv": 427,
             "openalex_citation_chasing_depth1": len(chase["works"])},
    n_after_dedup=len(records),
    n_title_abstract_excluded=int((ledger["verdict"] == "exclude").sum()),
    n_included=int((ledger["verdict"] == "include").sum()),
    n_rule_prefilter_excluded=int(arms.get("exclude", 0)),
    n_unresolved=int((ledger["verdict"] == "unclear").sum()),
    n_pre_window_deferred=int(arms.get("defer_pre_window", 0)),
    n_undecidable_at_prefilter=int(arms.get("unclear", 0)),
    fulltext_stage_performed=False)

print(json.dumps(flow["consistency"], indent=1))
assert flow["consistency"]["all_pass"], flow["consistency"]
print("\nscreened:", flow["screening"]["n_screened"],
      "= excluded", int((ledger.verdict == "exclude").sum()),
      "+ unresolved", int((ledger.verdict == "unclear").sum()),
      "+ included", int((ledger.verdict == "include").sum()))

{
 "screening_arms_sum_to_screened": true,
 "no_negative_counts": true,
 "all_pass": true
}

screened: 9090 = excluded 6614 + unresolved 943 + included 1533


In [9]:
inc = ledger[ledger["verdict"] == "include"]
by_channel = inc["channel"].value_counts().to_dict() if "channel" in inc.columns else {}
chasing = sum(v for k, v in by_channel.items() if str(k).startswith("cc"))
print("included by channel:", by_channel)
print(f"citation chasing contributed {chasing}/{len(inc)} = {chasing/len(inc):.0%} of the included corpus")

included by channel: {'cc_d1': 879, 'cc_d1_capfix': 375, 'c1': 279}
citation chasing contributed 1254/1533 = 82% of the included corpus


## 4. Claim extraction over the whole included corpus — REPLAYED

Claims were extracted from the abstracts of every included record by `claude-haiku-4-5`
(`charting.extract_claims`), one call per source, under the priority-ordered evidence-type rule
(magnitude before directional before taxonomic). The per-claim ledger
`outputs/extracted_claims_all_included.jsonl` is the persisted record; this cell reads it and
re-derives the reconciliation. With `REPLAY_LLM = True` the extraction re-executes against the
response cache instead.

In [10]:
from facetsearch import charting  # noqa: F401  (imported for the REPLAY path)

cl = pd.read_json(OUT / "extracted_claims_all_included.jsonl", lines=True)

if REPLAY_LLM:
    # Re-execute extraction against cache/live/: identical requests -> identical cached responses.
    from facetsearch import llm as llmmod
    clients = llmmod.build_clients(MODELS, "anthropic_openai_two_family",
                                   PRE["decoding"], CACHE / "live")
    by_key_all = {r.key: r for r in records}
    inc_all = [k for k in ledger.loc[ledger.verdict == "include", "key"] if k in by_key_all]
    wa = [k for k in inc_all if (by_key_all[k].abstract or "").strip()]
    src_rows = [{"key": k, "paper_id": k, "title": by_key_all[k].title or "",
                 "year": by_key_all[k].year, "venue": by_key_all[k].venue or "unknown",
                 "work_type": by_key_all[k].work_type or "unknown",
                 "text": (by_key_all[k].abstract or "").strip(),
                 "retrieval_depth": "abstract only"} for k in wa]
    cl = charting.extract_claims(clients["aux_charting_extraction"], src_rows, max_workers=8)

by_key = {r.key: r for r in records}
inc_keys = [k for k in ledger.loc[ledger.verdict == "include", "key"] if k in by_key]
with_abs = [k for k in inc_keys if (by_key[k].abstract or "").strip()]
real0 = cl[cl.claim_id.notna()]

print("included records:                 ", len(inc_keys))
print("with abstract -> extraction ran:  ", len(with_abs))
print("without abstract -> not extracted:", len(inc_keys) - len(with_abs))
print("sources yielding >=1 claim:       ", real0.paper_id.nunique(),
      "| yielding none:", len(with_abs) - real0.paper_id.nunique())
print("claims extracted:                 ", len(real0))
assert len(inc_keys) == N["corpus"]["n_included_total"]
assert len(with_abs) + (len(inc_keys) - len(with_abs)) == len(inc_keys)

included records:                  1533
with abstract -> extraction ran:   1446
without abstract -> not extracted: 87
sources yielding >=1 claim:        1404 | yielding none: 42
claims extracted:                  6075


## 5. Integrity screening — RE-DERIVED

Retracted sources are removed from the claim corpus before anything downstream sees it. Three
signals are unioned, because they disagree: the preregistered route (Crossref `update-to`
relations, queried per DOI and persisted to `outputs/retraction_status_all_included.csv`) detected
neither of the two retractions in this corpus, while OpenAlex's `is_retracted` flag and a marker
regex over deposited titles each detected both. The zero-sensitivity of the preregistered route is
itself a reported finding (`outputs/integrity_check.json`).

In [11]:
import re as _re

MARK = _re.compile(r"\b(retracted|retraction|withdrawn|expression of concern|"
                   r"editorial expression of concern|temporary removal)\b", _re.I)
rw_all = pd.read_csv(OUT / "retraction_status_all_included.csv")
xref_flag = set(rw_all.loc[rw_all.is_retracted == True, "doi"].astype(str).str.lower())
oa_flag = {k for k in inc_keys if bool(getattr(by_key[k], "is_retracted", False))}
title_flag = {k for k in inc_keys if MARK.search(by_key[k].title or "")}
union = oa_flag | title_flag | {k for k in inc_keys if (by_key[k].doi or "").lower() in xref_flag}

print("detection by source over the included corpus")
print("  Crossref update-to (preregistered):", len({k for k in inc_keys
      if (by_key[k].doi or "").lower() in xref_flag}))
print("  OpenAlex is_retracted:             ", len(oa_flag))
print("  title marker regex:                ", len(title_flag))
print("  union:                             ", len(union))
retracted_keys = sorted(union)
assert len(rw_all) == 1305, "the Crossref check must cover every DOI-bearing claim source"
assert oa_flag == title_flag, "the two positive signals identify the same records in this corpus"
print("\nunchecked residue: 99 sources carry no DOI; 9 DOIs did not resolve. The count is a "
      "lower bound\nand the check is re-run before submission (retractions accrue).")

detection by source over the included corpus
  Crossref update-to (preregistered): 0
  OpenAlex is_retracted:              2
  title marker regex:                 2
  union:                              2

unchecked residue: 99 sources carry no DOI; 9 DOIs did not resolve. The count is a lower bound
and the check is re-run before submission (retractions accrue).


## 6. The final claims table — RE-DERIVED

Claim rows join to source metadata on `paper_id`. Both sides carry `year`, so the join is written
with explicit suffixes and an assertion that every expected column exists — an earlier version
selected columns permissively and silently shipped the table without `year`. Claims from retracted
sources are excluded here, visibly: the excluded rows go to their own file rather than vanishing.

In [12]:
meta = pd.DataFrame([{"paper_id": k, "doi": by_key[k].doi, "title": by_key[k].title,
                      "year": by_key[k].year, "venue": by_key[k].venue,
                      "work_type": by_key[k].work_type} for k in inc_keys])
chan = dict(zip(ledger.key, ledger.channel))
merged = cl.merge(meta, on="paper_id", how="left", suffixes=("", "_meta"))
merged["year"] = merged["year"].fillna(merged["year_meta"])
merged["channel"] = merged.paper_id.map(chan)
merged["source_retracted"] = merged.paper_id.isin(retracted_keys)
merged["retraction_detected_by"] = np.where(merged.source_retracted, "openalex+title", "")

cols_final = ["claim_id", "paper_id", "doi", "title", "year", "venue", "work_type", "channel",
              "claim_text", "claim_role", "evidence_type", "quote", "retrieval_depth",
              "parse_ok", "source_retracted", "retraction_detected_by"]
missing = [c for c in cols_final if c not in merged.columns]
assert not missing, f"columns absent: {missing}"
final = merged[cols_final].sort_values(["paper_id", "claim_id"], na_position="last")
clean = final[~final.source_retracted]
real = clean[clean.claim_id.notna()]

print("claims before integrity exclusion:", int(final.claim_id.notna().sum()),
      "| excluded with retracted sources:", int(final.claim_id.notna().sum()) - len(real))
print("final claim corpus:", len(real), "claims from", real.paper_id.nunique(), "sources")
print("year populated:", int(real.year.notna().sum()), "| range:",
      int(real.year.min()), "-", int(real.year.max()))
print("by role:", real.claim_role.value_counts().to_dict())
print("by evidence type:", real.evidence_type.value_counts().to_dict())
assert len(real) == 6063 and real.paper_id.nunique() == 1402
assert real.year.notna().all(), "the year column must be populated on every claim"
assert int(final.claim_id.notna().sum()) - len(real) == 12

claims before integrity exclusion: 6075 | excluded with retracted sources: 12
final claim corpus: 6063 claims from 1402 sources
year populated: 6063 | range: 2015 - 2026
by role: {'signature': 4432, 'mechanism': 1631}
by evidence type: {'D': 3652, 'T': 1797, 'M': 614}


## 7. Validation of the claim extractor — RE-DERIVED

The extractor's evidence-type rule was validated on the 241-claim purposive-sample extraction that
preceded the corpus-wide pass: an independent model re-judged a stratified subsample, and the
priority-ordered definitions replaced glosses that had measured *below chance* (κ −0.207
→ 0.634, with the gain attributable to the extractor's definitions rather than codebook
alignment). This is inter-model reliability, not accuracy against a hand-read standard; the
subsample for that remains with the author.

In [13]:
claims = pd.read_csv(OUT / "extracted_claims.csv")
claims = claims[claims["claim_id"].notna()]
print("claims:", len(claims), "from", claims["paper_id"].nunique(), "sources")
print("by role:", claims["claim_role"].value_counts().to_dict())
print("by evidence type:", claims["evidence_type"].value_counts().to_dict())
print("every claim carries a quote:", bool(claims["quote"].notna().all()))

per_depth = claims.groupby(["retrieval_depth", "paper_id"]).size().groupby("retrieval_depth").mean()
print("\nmean claims per source by depth:", per_depth.round(2).to_dict())
print("-> mechanism-side evidence is thinner for reasons of RETRIEVAL, not of literature")

claims: 241 from 48 sources
by role: {'signature': 164, 'mechanism': 77}
by evidence type: {'D': 153, 'T': 67, 'M': 21}
every claim carries a quote: True

mean claims per source by depth: {'abstract only': 3.92, 'full text': 8.33}
-> mechanism-side evidence is thinner for reasons of RETRIEVAL, not of literature


In [14]:
val = json.loads((OUT / "extraction_validation_summary.json").read_text())
print("kappa, original definitions, vague validator: ", round(val["kappa_v1_vague_critic"], 3))
print("kappa, revised definitions, same validator:   ", round(val["kappa_vague_critic"], 3))
print("kappa, revised definitions, shared codebook:  ", round(val["kappa_shared_codebook"], 3))
print("quote supports claim:", val["quote_supported"])

# The two revised figures are near-identical, which is what establishes that the gain came from
# sharpening the EXTRACTOR's definitions rather than from aligning the two coders' rules. Had the
# shared-codebook figure been much the higher of the two, the reliability would be inflated by
# construction and could not be reported as evidence about the taxonomy.
gap = abs(val["kappa_shared_codebook"] - val["kappa_vague_critic"])
assert gap < 0.10, f"shared-codebook gain of {gap:.3f} would indicate inflation by construction"
print(f"\ngap between the two revised figures: {gap:.3f} -> not inflated by construction")

kappa, original definitions, vague validator:  -0.207
kappa, revised definitions, same validator:    0.634
kappa, revised definitions, shared codebook:   0.667
quote supports claim: 39/40

gap between the two revised figures: 0.033 -> not inflated by construction


## 8. Semantic frames: the 19-field predicate–argument schema — REPLAYED

Every claim carries one frame (`semantics.extract_frames`, `claude-haiku-4-5`): intervention and
its mode, affected construct, relation (mediates > moderates > causes > associates, priority-ordered),
direction and valence kept orthogonal, population, setting, horizon, magnitude, statistical marking
and epistemic strength as ordinals, and a verbatim evidence span. Frames were produced by the
synchronous route and the Anthropic Message Batches route; both key into one content-addressed
response cache, so the route leaves no trace in the data. Per-field reliability was measured on a
dual-extractor pilot with source-blocked bootstrap intervals; fields enter unsupervised analysis
only through `semantics.clustering_fields`, which reads the measured table rather than a hardcoded
list.

In [15]:
from facetsearch import semantics as sem

frames = pd.read_json(OUT.parent / "outputs" / "semantic_frames_all.jsonl", lines=True)
frames = frames.drop_duplicates(subset="claim_id", keep="first")
assert set(frames.claim_id) == set(real.claim_id), "frame set must match the claim set exactly"

joined = real.merge(frames.drop(columns=["paper_id"]), on="claim_id", how="left",
                    validate="one_to_one")
assert len(joined) == len(real) and joined.relation.notna().all()
print("frames:", len(frames), "| parse_ok:", round(float(frames.parse_ok.mean()), 4),
      "| evidence span verbatim in quote:", round(float(frames.span_verbatim.mean()), 4))

NULLS = {"not_stated", "not_applicable", "", None}
print("\npopulation rate per field (share of claims with a substantive value)")
for f in list(sem.NOMINAL_FIELDS) + list(sem.ORDINAL_FIELDS) + list(sem.FREE_FIELDS):
    col = joined[f]
    print(f"  {f:22s} {float((~col.isin(list(NULLS)) & col.notna()).mean()):.3f}")

rel = pd.read_csv(OUT / "sem_pilot_reliability_ci.csv")
print("\nreliability gate (source-blocked 95% bootstrap intervals, dual-extractor pilot)")
print(rel[["field", "population_rate", "score", "ci_lo", "ci_hi", "gate"]].to_string(index=False))
joined.to_csv(OUT / "claims_with_semantics.csv", index=False)
print("\nwrote outputs/claims_with_semantics.csv:", len(joined), "rows x", len(joined.columns), "columns")

frames: 6063 | parse_ok: 1.0 | evidence span verbatim in quote: 0.9617

population rate per field (share of claims with a substantive value)
  subject_type           0.819
  subject_mode           0.754
  object_type            0.850
  relation               0.862
  direction              0.807
  valence                0.831
  population_role        0.452
  population_setting     0.446
  measurement_horizon    0.159
  statistical_marking    0.199
  epistemic_strength     0.995
  subject_entity         0.992
  object_entity          1.000
  mediator               0.109
  moderator              0.101

reliability gate (source-blocked 95% bootstrap intervals, dual-extractor pilot)
              field  population_rate  score  ci_lo  ci_hi                                    gate
       subject_type            0.814  0.446  0.357  0.537          DROP — CI entirely below floor
       subject_mode            0.750  0.230  0.154  0.307          DROP — CI entirely below floor
        object_type

## 9. Provenance

In [16]:
print("configuration fingerprints")
for name in ("search_strategy", "preregistration", "seed_scheme", "models"):
    p = ROOT / "config" / f"{name}.yaml"
    print(f"  {name+'.yaml':26s} {hashlib.sha256(p.read_bytes()).hexdigest()[:16]}")

print("\nrole bindings")
for role, mk in MODELS["role_bindings"]["anthropic_openai_two_family"].items():
    print(f"  {role:26s} {MODELS['models'][mk]['model_id']}  ({MODELS['models'][mk]['provider']})")

print("\ncost")
print("  reported run total  $", N["cost"]["total_usd"])
print("  ceiling             $", N["cost"]["ceiling_usd"])
print("\nDeviations in force are listed in METHODS_IMPLEMENTED.md §17 (D-1 through D-18).")
print("Two are load-bearing for how the results above must be read:")
print("  D-1  both coders share one vendor, so reported α is an UPPER BOUND on independent-coder α")
print("  D-12 residual converted from a feasibility gate into a search objective")

configuration fingerprints
  search_strategy.yaml       2a4b59258cd2b551
  preregistration.yaml       60fa2b8a572c281b
  seed_scheme.yaml           beb7d0d1c483bb41
  models.yaml                4b7a50ae4bdd484e

role bindings
  coder_a                    gpt-5.6-luna  (openai)
  coder_b                    gpt-5-mini  (openai)
  critic                     claude-sonnet-5  (anthropic)
  aux_semantic_retrieval     gpt-5.6-luna  (openai)
  aux_charting_extraction    claude-haiku-4-5  (anthropic)

cost
  reported run total  $ 6.81
  ceiling             $ 10.0

Deviations in force are listed in METHODS_IMPLEMENTED.md §17 (D-1 through D-18).
Two are load-bearing for how the results above must be read:
  D-1  both coders share one vendor, so reported α is an UPPER BOUND on independent-coder α
  D-12 residual converted from a feasibility gate into a search objective


In [17]:
print("outputs regenerated or asserted by this notebook:")
for f in sorted((ROOT / "outputs").glob("*")):
    print(f"  {f.name:38s} {f.stat().st_size:>9,d} bytes")

outputs regenerated or asserted by this notebook:
  _not_from_this_run                           160 bytes
  algorithm1_halt_finding.json               2,182 bytes
  alpha_diagnosis.csv                          305 bytes
  alpha_table_S0.csv                           338 bytes
  author_axis_matrix.csv                     3,602 bytes
  cca.json                                     280 bytes
  channel_yield.csv                            121 bytes
  charted_sources.csv                       63,031 bytes
  claims_excluded_retracted.csv              6,749 bytes
  claims_with_semantics.csv              5,265,331 bytes
  codings_S0.csv                           260,776 bytes
  codings_holdout.csv                       94,740 bytes
  consensus_codes_S0.csv                    19,366 bytes
  cost_envelope.csv                            321 bytes
  dsl_estimates.csv                          1,298 bytes
  extracted_claims.csv                      84,089 bytes
  extracted_claims_all_included.csv   

## What this notebook establishes

Given the persisted ledgers, every number in the paper's corpus and claims sections follows
deterministically, and the assertions above fail loudly if any of them drifts: the prefilter arms
sum to the deduplicated corpus; the screened set equals the prefilter-passing set; no record was
included on a title-only judgement; the PRISMA arms reconcile; the claim table reconciles to the
included corpus (1,533 = 1,446 extracted + 87 without abstracts); claims from retracted sources
are excluded and counted; and the 6,063 final claims join one-to-one with their semantic frames.
What it does not establish is extraction accuracy against a hand-read standard — the reported
figures are inter-model reliability — and the `relation` field's reliability under the final
prompt remains unmeasured (last measured κ 0.532 [0.433, 0.627] under the prior prompt).